In [24]:
# ml_train_ranker — Cross-sectional weekly ranking with LightGBM Ranker (+Ridge ensemble)

from pathlib import Path
import pandas as pd
import numpy as np
import json, re, math, time
from dataclasses import dataclass, asdict

# 可選：如未安裝 LightGBM，先安裝（已裝則略過）
# %pip install lightgbm==4.3.0

# 可選：Ridge 做為穩定 baseline
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr

ROOT = Path("..")
CUR  = ROOT / "data" / "curated"
MET  = ROOT / "data" / "metrics"
MODEL = ROOT / "models"
for p in [CUR, MET, MODEL]:
    p.mkdir(parents=True, exist_ok=True)

@dataclass
class CFG:
    lookback_weeks: int = 52        # 起始訓練窗
    min_assets_per_week: int = 10   # 週最少資產數
    asset_cap: int = 30             # 每週最多資產數（Top30）
    early_stop_weeks: int = 8       # 最近 N 週為 validation（早停）
    seed: int = 42

    # LightGBM Ranker（保守參數）
    lgbm_params: dict = None

    # Ridge baseline
    ridge_alpha: float = 1.0

    # Ensemble 權重（ranker + ridge）
    w_ranker: float = 0.7
    w_ridge: float  = 0.3

    # 輸出
    out_pred: Path = CUR / "ml_preds_weekly.parquet"
    out_log:  Path = MET / f"ml_train_logs_{pd.Timestamp.utcnow().strftime('%Y%m%d')}.json"
    out_imp:  Path = MET / f"ml_feature_importance_{pd.Timestamp.utcnow().strftime('%Y%m%d')}.csv"

cfg = CFG(
    lgbm_params=dict(
        objective = "lambdarank",
        metric    = "ndcg",
        ndcg_eval_at = [5,10,30],
        boosting_type = "gbdt",
        n_estimators = 3000,
        learning_rate = 0.03,
        num_leaves = 63,         # 保守
        max_depth = -1,
        min_data_in_leaf = 20,
        subsample = 0.8,
        colsample_bytree = 0.8,
        reg_alpha = 0.0,
        reg_lambda = 1.0,
        random_state = 42,
        verbose = -1,
        force_row_wise = True,
        label_gain = [0, 1, 3, 7, 15]   
    )
)

print(cfg)


CFG(lookback_weeks=52, min_assets_per_week=10, asset_cap=30, early_stop_weeks=8, seed=42, lgbm_params={'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [5, 10, 30], 'boosting_type': 'gbdt', 'n_estimators': 3000, 'learning_rate': 0.03, 'num_leaves': 63, 'max_depth': -1, 'min_data_in_leaf': 20, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.0, 'reg_lambda': 1.0, 'random_state': 42, 'verbose': -1, 'force_row_wise': True, 'label_gain': [0, 1, 3, 7, 15]}, ridge_alpha=1.0, w_ranker=0.7, w_ridge=0.3, out_pred=PosixPath('../data/curated/ml_preds_weekly.parquet'), out_log=PosixPath('../data/metrics/ml_train_logs_20251001.json'), out_imp=PosixPath('../data/metrics/ml_feature_importance_20251001.csv'))


In [25]:
# 讀取週資料與宇宙
df  = pd.read_parquet(CUR / "factors_weekly.parquet")
uni = pd.read_parquet(CUR / "universe_top30_annual.parquet")

df["date_week"] = pd.to_datetime(df["date_week"]).dt.tz_localize(None)
uni["symbol"] = uni["symbol"].astype(str).str.upper()

# 資產鍵
ASSET_KEYS = ["market","symbol","coingecko_id"]
asset_key = next((k for k in ASSET_KEYS if k in df.columns), None)
if asset_key is None:
    asset_key = "_asset"; df["_asset"] = "asset_0"
if "symbol" in df.columns:
    df["symbol"] = df["symbol"].astype(str).str.upper()

# 目標：下一週的超額報酬
if "rf_weekly" not in df.columns:
    df["rf_weekly"] = 0.0
df = df.sort_values([asset_key,"date_week"]).copy()
df["excess"] = df["ret_simple_weekly"] - df["rf_weekly"]
df["y_fwd1"] = df.groupby(asset_key)["excess"].shift(-1)

# 特徵集合（優先 *_z；若無則退回原值）
base_cols = [
    "log_mcap_year","log_price","max_price_week",
    "r1","r2","r3","r4","r4_1","rmom3",
    "prcvol_mean_week","prcvol_std_week","vol_4w"
]
fac_cols = [c+"_z" for c in base_cols if c+"_z" in df.columns]
if not fac_cols:
    fac_cols = [c for c in base_cols if c in df.columns]
print("features:", fac_cols)

# 以年匹配的 Top30 限定宇宙
df["year"] = df["date_week"].dt.year
df_u = df.merge(
    uni[["year","symbol"]].drop_duplicates(),
    on=["year","symbol"],
    how="inner"
)

# 簡單 winsorize（可選）再 z-score 已做，這裡不重複；缺值處理在每週過濾
print("rows (universe matched):", len(df_u))
df_u.head(3)


features: ['log_mcap_year_z', 'log_price_z', 'max_price_week_z', 'r1_z', 'r2_z', 'r3_z', 'r4_z', 'r4_1_z', 'rmom3_z', 'prcvol_mean_week_z', 'prcvol_std_week_z', 'vol_4w_z']
rows (universe matched): 5650


,date_week,market,symbol,close,ret_simple_weekly,log_mcap_year,log_price,max_price_week,r1,r2,...,r4_z,r4_1_z,rmom3_z,prcvol_mean_week_z,prcvol_std_week_z,vol_4w_z,rf_weekly,excess,y_fwd1,year
0,2018-04-29,ADA/USDT,ADA,0.36254,0.287429,23.649626,0.309351,0.3866,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.287429,-0.044133,2018
1,2018-05-06,ADA/USDT,ADA,0.34654,-0.044133,23.649626,0.297538,0.3885,0.287429,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-0.044133,-0.200150,2018
2,2018-05-13,ADA/USDT,ADA,0.27718,-0.200150,23.649626,0.244655,0.3474,-0.044133,0.230611,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,-0.200150,-0.078000,2018


In [26]:
def weekly_tradable(g: pd.DataFrame) -> pd.DataFrame:
    g2 = g.dropna(subset=["y_fwd1"] + fac_cols)
    if len(g2) > cfg.asset_cap:
        if "log_mcap_year" in g2.columns:
            g2 = g2.sort_values("log_mcap_year", ascending=False).head(cfg.asset_cap)
        elif "log_price" in g2.columns:
            g2 = g2.sort_values("log_price", ascending=False).head(cfg.asset_cap)
        else:
            g2 = g2.head(cfg.asset_cap)
    return g2

blocks = []
for d, g in df_u.groupby("date_week"):
    gg = weekly_tradable(g)
    if len(gg) >= cfg.min_assets_per_week:
        blocks.append(gg)

df_tr = pd.concat(blocks, ignore_index=True).sort_values(["date_week", asset_key])
print("filtered rows:", len(df_tr), "weeks:", df_tr['date_week'].nunique(), "assets:", df_tr[asset_key].nunique())
df_tr.head(3)


filtered rows: 5420 weeks: 377 assets: 33


,date_week,market,symbol,close,ret_simple_weekly,log_mcap_year,log_price,max_price_week,r1,r2,...,r4_z,r4_1_z,rmom3_z,prcvol_mean_week_z,prcvol_std_week_z,vol_4w_z,rf_weekly,excess,y_fwd1,year
0,2018-07-08,ADA/USDT,ADA,0.14481,0.018999,23.649626,0.135239,0.16209,0.085389,-0.114248,...,-0.380990,-0.380990,0.775004,-0.432281,-0.682654,0.213359,0.0,0.018999,-0.017540,2018
1,2018-07-08,BTC/USDT,BTC,6712.10000,0.055891,26.193292,8.811816,6818.16000,0.035822,-0.014388,...,1.277492,1.277492,0.812162,2.862730,1.389997,-1.412823,0.0,0.055891,-0.053499,2018
2,2018-07-08,EOS/USDT,EOS,8.68270,0.069772,22.340998,2.270341,9.43630,0.006998,-0.214957,...,-0.958801,-0.958801,-0.434406,0.596316,0.301612,0.543625,0.0,0.069772,-0.146867,2018


In [28]:
from scipy.stats import spearmanr

def build_group_sizes(frame: pd.DataFrame) -> np.ndarray:
    grp = frame.groupby("date_week", sort=True).size()
    arr = grp.to_numpy(dtype=np.int32)
    assert int(arr.sum()) == int(len(frame)), "Group sizes do not sum to number of rows."
    return arr

def rankic(y_hat, y_true):
    return spearmanr(y_hat, y_true, nan_policy="omit").correlation

def safe_filename(s: str) -> str:
    import re
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s)

def to_relevance_levels(df: pd.DataFrame, n_levels: int = 5, y_col: str = "y_fwd1", group_col: str = "date_week") -> np.ndarray:
    """
    將連續的 y（每週一個 group）轉成 0..(n_levels-1) 的整數等級。
    主要用於 LightGBM lambdarank 的 label。
    """
    rel = np.empty(len(df), dtype=np.int32)
    # 依每週（group）分別做分位數分桶
    order = df[group_col].values
    for w, idx in df.groupby(group_col).indices.items():
        y = df.loc[idx, y_col].values.astype(float)
        # 若全部 NaN 或樣本太少，全部給中間等級
        if np.all(np.isnan(y)) or len(y) < 2:
            rel[idx] = n_levels // 2
            continue
        # 若唯一值過少，改用秩百分位
        uniq = np.unique(y[~np.isnan(y)])
        if len(uniq) < n_levels:
            # rank method='average'，轉成 0~1，再切成 n_levels 份
            ranks = pd.Series(y).rank(method="average", na_option="keep")
            pct = (ranks - 1) / (len(ranks.dropna()) - 1) if len(ranks.dropna()) > 1 else pd.Series(np.full_like(ranks, 0.5))
            bins = np.floor(pct * n_levels).clip(0, n_levels - 1)
            rel[idx] = bins.fillna(n_levels // 2).astype(np.int32).values
        else:
            # 正常用 qcut
            try:
                q = pd.qcut(y, q=n_levels, labels=False, duplicates="drop")
                # 若 duplicates 導致層級 < n_levels，補齊到 0..levels-1 的範圍
                maxlev = int(np.nanmax(q))
                if maxlev + 1 < n_levels:
                    # 線性拉伸到 0..(n_levels-1)
                    q = (q.astype(float) * (n_levels - 1) / max(maxlev, 1)).round().astype("Int64")
                rel[idx] = q.fillna(n_levels // 2).astype(np.int32).values
            except Exception:
                # 退回秩百分位法
                ranks = pd.Series(y).rank(method="average", na_option="keep")
                pct = (ranks - 1) / (len(ranks.dropna()) - 1) if len(ranks.dropna()) > 1 else pd.Series(np.full_like(ranks, 0.5))
                bins = np.floor(pct * n_levels).clip(0, n_levels - 1)
                rel[idx] = bins.fillna(n_levels // 2).astype(np.int32).values
    return rel


In [31]:
import lightgbm as lgb

all_dates = sorted(df_tr["date_week"].unique())
pred_rows, log_items = [], []

for i, d in enumerate(all_dates):
    hist_dates = all_dates[:i]
    if len(hist_dates) < cfg.lookback_weeks:
        continue

    train_end = hist_dates[-1]
    train = df_tr[df_tr["date_week"] <= train_end].copy()
    test  = df_tr[df_tr["date_week"] == d].copy()
    if len(test) < cfg.min_assets_per_week:
        continue

    tr_weeks = sorted(train["date_week"].unique())
    if len(tr_weeks) <= cfg.early_stop_weeks + 1:
        continue
    valid_start = tr_weeks[-cfg.early_stop_weeks]

    tr = train[train["date_week"] <  valid_start].copy().reset_index(drop=True)
    va = train[train["date_week"] >= valid_start].copy().reset_index(drop=True)


    # ---- 關鍵：將 y 轉成 lambdarank 的等級標籤（整數）----
    y_tr_rel = to_relevance_levels(tr, n_levels=5, y_col="y_fwd1", group_col="date_week")
    y_va_rel = to_relevance_levels(va, n_levels=5, y_col="y_fwd1", group_col="date_week")

    X_tr, X_va = tr[fac_cols].values, va[fac_cols].values
    X_te       = test[fac_cols].values
    g_tr = build_group_sizes(tr)
    g_va = build_group_sizes(va)
    assert g_tr.dtype == np.int32 and g_va.dtype == np.int32

    dtr = lgb.Dataset(X_tr, label=y_tr_rel, group=g_tr, free_raw_data=True)
    dva = lgb.Dataset(X_va, label=y_va_rel, group=g_va, free_raw_data=True)

    params = dict(cfg.lgbm_params)
    params["random_state"] = cfg.seed

    booster = lgb.train(
        params,
        dtr,
        valid_sets=[dtr, dva],
        valid_names=["train","valid"],
        num_boost_round=params.get("n_estimators", 3000),
        callbacks=[
            lgb.early_stopping(stopping_rounds=max(20, cfg.early_stop_weeks*2), verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    # 推論：仍對 test 的特徵輸出連續分數（用於排序）
    y_hat_ranker = booster.predict(X_te, num_iteration=booster.best_iteration)

    # Ridge baseline 用原始連續 y 訓練（tr+va 合併）
    from sklearn.linear_model import Ridge
    ridge = Ridge(alpha=cfg.ridge_alpha, fit_intercept=True, random_state=cfg.seed)
    ridge.fit(
        np.vstack([X_tr, X_va]),
        np.concatenate([tr["y_fwd1"].values, va["y_fwd1"].values])
    )
    y_hat_ridge = ridge.predict(X_te)

    # Ensemble
    y_pred = cfg.w_ranker * y_hat_ranker + cfg.w_ridge * y_hat_ridge

    out = test[["date_week", asset_key, "y_fwd1"]].copy()
    out["y_pred_ranker"] = y_hat_ranker
    out["y_pred_ridge"]  = y_hat_ridge
    out["y_pred"]        = y_pred
    pred_rows.append(out)

    # 記錄：用「連續 y 的 RankIC」衡量（與投組層一致）
    # 這裡的驗證集用 ranker 對連續 y 的排序力做監控
    y_tr_pred = booster.predict(X_tr, num_iteration=booster.best_iteration)
    y_va_pred = booster.predict(X_va, num_iteration=booster.best_iteration)
    log_items.append({
        "asof_week": d.strftime("%Y-%m-%d"),
        "best_iter": int(booster.best_iteration),
        "train_rankic": float(rankic(y_tr_pred, tr["y_fwd1"].values)),
        "valid_rankic": float(rankic(y_va_pred, va["y_fwd1"].values)),
        "params": cfg.lgbm_params
    })

pred = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
print("pred rows:", len(pred))
pred.head(5)


/opt/anaconda3/envs/crypto_env/lib/python3.13/site-packages/lightgbm/engine.py:172: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
/opt/anaconda3/envs/crypto_env/lib/python3.13/site-packages/lightgbm/engine.py:172: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
/opt/anaconda3/envs/crypto_env/lib/python3.13/site-packages/lightgbm/engine.py:172: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
/opt/anaconda3/envs/crypto_env/lib/python3.13/site-packages/lightgbm/engine.py:172: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")
/opt/anaconda3/envs/crypto_env/lib/python3.1

pred rows: 4754


,date_week,market,y_fwd1,y_pred_ranker,y_pred_ridge,y_pred
0,2019-07-07,ADA/USDT,-0.256838,0.028285,-0.000728,0.019581
1,2019-07-07,BNB/USDT,-0.133286,0.006538,0.012896,0.008445
2,2019-07-07,BTC/USDT,-0.108016,0.031711,0.004640,0.023590
3,2019-07-07,DASH/USDT,-0.219083,-0.044817,0.003371,-0.030360
4,2019-07-07,EOS/USDT,-0.308344,-0.067749,0.005372,-0.045813


In [32]:
if pred.empty:
    raise SystemExit("No predictions produced. Check filters/parameters.")

# Overall RankIC（ensemble / ranker / ridge）
overall = {
    "ensemble_RankIC": spearmanr(pred["y_pred"], pred["y_fwd1"], nan_policy="omit").correlation,
    "ranker_RankIC":   spearmanr(pred["y_pred_ranker"], pred["y_fwd1"], nan_policy="omit").correlation,
    "ridge_RankIC":    spearmanr(pred["y_pred_ridge"], pred["y_fwd1"], nan_policy="omit").correlation,
    "obs": int(len(pred))
}
print("Overall:", overall)

# 每週 RankIC（ensemble）
rows = []
for d, g in pred.groupby("date_week"):
    if len(g) >= cfg.min_assets_per_week:
        rows.append({"date_week": d, "rankIC": spearmanr(g["y_pred"], g["y_fwd1"]).correlation})
ic_by_week = pd.DataFrame(rows).sort_values("date_week")
display(ic_by_week.describe())

# 存檔（預測）
pred.to_parquet(cfg.out_pred, index=False)
print("saved preds:", cfg.out_pred)

# 存檔（訓練紀錄）
log_rec = {
    "config": asdict(cfg),
    "overall": overall,
    "timeline": log_items
}
with open(cfg.out_log, "w") as f:
    json.dump(log_rec, f, indent=2, default=str)
print("saved log:", cfg.out_log)


Overall: {'ensemble_RankIC': np.float64(-0.02253998169911864), 'ranker_RankIC': np.float64(0.0429610587167301), 'ridge_RankIC': np.float64(-0.038300097733458), 'obs': 4754}


,date_week,rankIC
count,325,325.000000
mean,2022-08-14 00:00:00,-0.041481
min,2019-07-07 00:00:00,-0.850000
25%,2021-01-24 00:00:00,-0.257143
50%,2022-08-14 00:00:00,-0.032967
75%,2024-03-03 00:00:00,0.173626
max,2025-09-21 00:00:00,0.764286
std,NaN,0.297809


saved preds: ../data/curated/ml_preds_weekly.parquet
saved log: ../data/metrics/ml_train_logs_20251001.json


In [33]:
# 取最後一次 booster 的重要度（也可在 loop 中每次保存）
try:
    import lightgbm as lgb
    # 如果上面還保留最後一輪的 booster 物件
    imp_gain = booster.feature_importance(importance_type="gain")
    imp_split = booster.feature_importance(importance_type="split")
    imp = pd.DataFrame({
        "feature": fac_cols,
        "gain": imp_gain,
        "split": imp_split
    }).sort_values("gain", ascending=False)
    display(imp.head(20))
    imp.to_csv(cfg.out_imp, index=False)
    print("saved importance:", cfg.out_imp)
except Exception as e:
    print("importance not available:", e)


,feature,gain,split
5,r3_z,435.771420,79
11,vol_4w_z,414.475752,65
9,prcvol_mean_week_z,360.992839,64
4,r2_z,348.955089,52
3,r1_z,308.185091,63
2,max_price_week_z,300.769980,45
10,prcvol_std_week_z,275.300279,44
6,r4_z,274.191439,54
1,log_price_z,269.613411,45
8,rmom3_z,265.450251,46


saved importance: ../data/metrics/ml_feature_importance_20251001.csv
